In [ ]:
# papermill parameters

# i/o
input_ohlcv_file='usdt.parquet'
input_gjr_file='garch.parquet'

output_feat_file='chronos-2-features.parquet'
output_pred_file='pred.parquet'
output_eval_file='chronos-2-eval.parquet'

# chronos-2
device_map = 'cpu'
cross_learning = True
lookback = 60
lookforward = 1
norm_axis = 'rows' # rows: time series, columns: cross-sectional

reference_pair = 'BTCUSDT'

# derived, forecasted metrics
gjr_garch_dist = 'skewt'

In [ ]:
print(f'''
input_ohlcv_file = {input_ohlcv_file}
input_gjr_file = {input_gjr_file}

output_feat_file = {output_feat_file}
output_pred_file = {output_pred_file}
output_eval_file = {output_eval_file}

device_map = {device_map}
cross_learning = {cross_learning}
lookback = {lookback}
lookforward = {lookforward}

# derived, forecasted metrics
gjr_garch_dist = {gjr_garch_dist}
''')

In [ ]:
# volatility, taker buy/sell ratio, momentum, auto correlation, level

import polars as pl
from arch import arch_model
import numpy as np
from tqdm import tqdm

ohlcv = pl.read_parquet(input_ohlcv_file).sort(['symbol', 'ts'])
gjr = pl.read_parquet(input_gjr_file).sort(['symbol', 'ts'])

df = (ohlcv
    .join(gjr, on=['symbol','ts'])
    .filter(
        (pl.col('open') > 0) & (pl.col('high') > 0) &
        (pl.col('low') > 0) & (pl.col('close') > 0) &
        (pl.col('base_volume') > 0) & 
        (pl.col('quote_volume') > 0))
    .with_columns([
        # todays log return (target)
        (pl.col('close') / pl.col('close').shift(1))
            .log()
            .over('symbol')
            .alias('ret')])
    .select([
        # base columns
        pl.col('ts'),
        pl.col('symbol'),
        pl.col('ret'),

        # log wick symmetry
        ((pl.col('high') - pl.col('close') + pl.lit(1e-6)) /
         (pl.col('close') - pl.col('low') + pl.lit(1e-6)))
            .log()
            .over('symbol')
            .alias('sym'),

        pl.col('quote_volume'),
    
        # rogers-satchell volatility
        (((pl.col('high') / pl.col('close')).log() * (pl.col('high') / pl.col('open')).log()) +
         ((pl.col('low') / pl.col('close')).log() * (pl.col('low') / pl.col('open')).log()))
            .sqrt()
            .alias('sigma_rs'),
    
        (pl.col('quote_volume') / pl.col('base_volume')).alias('vwap'),
        ((2 * pl.col('taker_buy_base_volume') / pl.col('base_volume')) - 1)
            .alias('ofi'),
    
        # n-day momentum
        (pl.col('ret').rolling_sum(window_size=30)
            .over('symbol')
            .alias('momentum')),
    
        # gjr-garch
        pl.col('forecast').alias('sigma_forecast'),
        pl.col('mu'), pl.col('omega'), pl.col('alpha[1]'), pl.col('gamma[1]'),
        pl.col('beta[1]'), pl.col('eta'), pl.col('lambda'), 
    ])
    .drop_nulls()
    .write_parquet(output_feat_file))

In [ ]:
# inference with chronos-2

import numpy as np
import pandas as pd
from chronos import Chronos2Pipeline
import datetime as dt
from tqdm import tqdm

pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=device_map)
pred = []
df = pl.read_parquet(output_feat_file)

for td in tqdm(df['ts'].unique().sort()):
    lb = td - dt.timedelta(days=lookback)

    # extract batch
    win = (df
        .filter((pl.col('ts') >= lb) & (pl.col('ts') <= td))
        .drop_nulls()
        .upsample(time_column="ts", every="1d", group_by="symbol")
        .with_columns(pl.all().forward_fill())
        .filter(
            (pl.len().over("symbol") >= lookback) & (pl.col("ts").max().over("symbol") == td))
    )

    if len(win) < 3:
        continue

    future_df = pl.DataFrame()
    
    # forecast volatility
    for sym in win['symbol'].unique():
        row = df.filter((pl.col('ts') == td) & (pl.col('symbol') == sym))
        if row.height != 1:
            continue
        if row['sigma_forecast'].is_nan().all():
            var = np.full(lookforward, np.nan)
        else:
            params = np.array([
                row['mu'][0],
                row['omega'][0],
                row['alpha[1]'][0],
                row['gamma[1]'][0],
                row['beta[1]'][0],
                row['eta'][0],
                row['lambda'][0]
            ])
            ret = win.filter(pl.col('symbol') == sym).select('ret').to_numpy().flatten()
            fc = arch_model(
                    ret * 100,
                    vol='Garch',
                    p=1,
                    o=1,
                    q=1,
                    dist=gjr_garch_dist,
                    rescale=False
                ).forecast(params=params, horizon=lookforward, reindex=False)
            var = np.sqrt(fc.variance.values[-1, :]) / 100

        future_df = pl.concat([future_df, pl.DataFrame({
            'ts': [td + dt.timedelta(days=d+1) for d in range(lookforward)],
            'sigma_forecast': var,
            'symbol': sym
        })])

    # normalize batch
    features = [c for c in win.columns if c not in ['ts', 'symbol', 'ret']]
    if norm_axis == 'rows':
        norm = win.with_columns([
            ((pl.col(c) - pl.col(c).mean().over('symbol')) /
             (pl.col(c).std().over('symbol') + 1e-9)).alias(c)
            for c in features])
        future_norm = future_df.with_columns([
            ((pl.col(c) - pl.col(c).mean().over('symbol')) /
             (pl.col(c).std().over('symbol') + 1e-9)).alias(c)
            for c in ['sigma_forecast']])
    elif norm_axis == 'columns':
        norm = win.with_columns([
            ((pl.col(c) - pl.col(c).mean().over("ts")) / 
             (pl.col(c).std().over("ts") + 1e-9)).alias(c)
            for c in features])
        future_norm = win.with_columns([
            ((pl.col(c) - pl.col(c).mean().over("ts")) / 
             (pl.col(c).std().over("ts") + 1e-9)).alias(c)
            for c in ['sigma_forecast']])
    else:
        raise Exception(f'unknown normalization direction {norm_axis}')
    
    # Generate predictions with covariates
    pdf = pipeline.predict_df(
        norm.sort(['ts','symbol']).to_pandas(),
        future_norm.sort(['ts','symbol']).to_pandas(),
        prediction_length=lookforward,
        quantile_levels=[0.5],
        id_column="symbol",
        timestamp_column="ts",
        target="ret",
        validate_inputs=True,
        cross_learning=cross_learning,
    )

    if pdf['0.5'].isna().any():
        raise Exception(f'NaN prediction for batch {td}')

    pdf = (
        pl.from_pandas(pdf)
        .group_by("symbol")
        .agg([
            pl.col("0.5").sum().alias("prediction"),
            pl.col("ts").max().alias("horizon")
        ])
        .with_columns(pl.lit(td).alias("ts"))
    )

    pred.append(pdf)

pl.concat(pred).write_parquet(output_pred_file)

In [ ]:
# compute performance of a long only portfolio of the top decile coins, weighted by USD volume, 1d holing time

df = (pl
    .read_parquet(output_feat_file)
    .with_columns(pl.col('ts').dt.cast_time_unit("ms"))
    
    # join chronos-2 predictions
    .join(pl.read_parquet(output_pred_file)
            .select([
                pl.col('ts').dt.cast_time_unit("ms"),
                pl.col('symbol'),
                pl.col('prediction').alias('pred')
            ]),
          on=['ts','symbol'],
          how='inner')
    
    # join closing prices
    .join(pl.read_parquet(input_ohlcv_file)
            .select([pl.col('ts'),pl.col('symbol'),pl.col('close')]),
          on=['ts','symbol'],
          how='inner')
        
    .sort(['ts','symbol'])

    # filter some extreme events like the Terra Luna
    .filter(pl.col('ret') < np.sqrt(5))

    # sort predictions in decile
    .with_columns([
        pl.col('pred')
            .qcut(
                10, 
                allow_duplicates=True, 
                labels=list(map(lambda n: f'{n}', range(1,11))))
            .over('ts')
            .alias('rank'),
        pl.col('quote_volume').alias('avg_usd_vol')
    ])

    # keep top decile (long-only)
    .filter([
        pl.col('rank') == '10', pl.col('pred') > 0])

    # weight by cross-sectional daily USD volume
    .with_columns([
        (pl.col('avg_usd_vol') / pl.col('avg_usd_vol').sum().over(['ts'])).alias('weight')])

    # remove all weights under 5%
    .filter(pl.col('weight') > 0.05)
    .with_columns([
        (pl.col('weight') / pl.col('weight').sum().over(['ts'])).alias('weight')])
    
    # hold one day
    .with_columns([
        pl.col('weight').alias('initial'),
        (pl.col('weight') * pl.col('ret').exp()).alias('eod')])

    # incorporate 0.2% tx fee
    .with_columns([
        ((pl.col('eod').abs() + pl.col('initial').abs()) * 0.002).alias('fee')])
    .with_columns([
        (pl.col('eod') - pl.col('initial') - pl.col('fee')).alias('pnl')]))

# check df before continuing
wc = (df
    .group_by("ts")
    .agg(pl.col("weight").sum().alias("total_weight"))
    .filter((pl.col("total_weight") - 1.0).abs() > 1e-6))
if not wc.is_empty():
    print(f"Weight sum error at timestamps: {wc['ts'].to_list()}")

(df
    # aggregate per-coin returns to daily portfolio returns
    .sort('ts')
    .group_by('ts')
    .agg([
        (1 + pl.col('pnl').sum()).log().alias('strategy'),
        pl.col('fee').sum().alias('fee'),])

    # join btc as benchmark
    .join(pl.read_parquet(output_feat_file)
            .filter(pl.col('symbol') == reference_pair)
            .select([pl.col('ts'),pl.col('ret').alias('ref')]),
        on='ts',
        how='inner')
    .with_columns([
        (pl.col('strategy') - pl.col('ref')).alias('perf')])

    # compute the equity curve staring with 0
    .with_columns([
        (pl.col('strategy').cum_sum()).alias('equity')])

    # write out results
    .write_parquet(output_eval_file))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
from datetime import datetime, timedelta
import pytz

# 1. Setup Timezones & Current Time
malaysia_tz = pytz.timezone("Asia/Kuala_Lumpur")
current_time_myt = datetime.now(malaysia_tz).strftime('%Y-%m-%d %H:%M:%S')

latest_ts = df['ts'].max()
start_ts = latest_ts - timedelta(days=lookback)

# 2. Data Extraction
portfolio = df.filter(pl.col('ts') == latest_ts).sort('weight', descending=True).to_pandas()
top_symbols = portfolio['symbol'].tolist() 

context_data = (
    pl.read_parquet(output_feat_file)
    .filter((pl.col('symbol').is_in(top_symbols[:5])) & (pl.col('ts') >= start_ts) & (pl.col('ts') <= latest_ts))
    .with_columns([
        pl.col('ts').dt.replace_time_zone("UTC").dt.convert_time_zone("Asia/Kuala_Lumpur").alias('ts_local'),
        (pl.col('ret').cum_sum().over('symbol')).alias('cum_ret')
    ])
    .to_pandas()
)

# 3. Visualization Setup
fig = plt.figure(figsize=(22, 20)) # Tall format for better table spacing
gs = fig.add_gridspec(4, 2, height_ratios=[1.2, 1, 1, 0.6]) 
sns.set_style("darkgrid")

# --- A. Full Lookback History ---
ax1 = fig.add_subplot(gs[0, :])
sns.lineplot(data=context_data, x='ts_local', y='cum_ret', hue='symbol', ax=ax1, palette="husl", lw=3)
ax1.set_title(f"Market Context: {lookback}-Day Cumulative Return History", fontsize=20, fontweight='bold')
ax1.set_ylabel("Cumulative Log Return", fontsize=15)

# --- B. Allocation Donut ---
ax2 = fig.add_subplot(gs[1:3, 0])
ax2.pie(portfolio['weight'], labels=portfolio['symbol'], autopct='%1.1f%%', 
        startangle=140, colors=sns.color_palette("rocket_r", n_colors=len(portfolio)), 
        pctdistance=0.8, explode=[0.05]*len(portfolio), textprops={'fontsize': 16})
ax2.add_artist(plt.Circle((0,0), 0.70, fc='white'))
ax2.set_title("Target Portfolio Weights", fontsize=20, fontweight='bold')

# --- C. Expected Returns ---
ax3 = fig.add_subplot(gs[1, 1])
sns.barplot(data=portfolio, x='symbol', y='pred', hue='symbol', ax=ax3, palette="viridis", legend=False)
ax3.set_title("Chronos-2: Forecasted 1-Day Return", fontsize=18)
ax3.axhline(0, color='red', linestyle='--', alpha=0.5)

# --- D. Feature Profile ---
ax4 = fig.add_subplot(gs[2, 1])
current_feats = df.filter(pl.col('ts') == latest_ts).to_pandas()
ax4.scatter(data=current_feats, x='sigma_rs', y='momentum', 
            s=current_feats['weight'] * 25000, c=current_feats['pred'], 
            cmap='RdYlGn', alpha=0.7, edgecolors="w")
for i, txt in enumerate(current_feats['symbol']):
    ax4.annotate(txt, (current_feats['sigma_rs'][i], current_feats['momentum'][i]), 
                 fontsize=14, fontweight='bold', xytext=(5,5), textcoords='offset points')
ax4.set_title("Strategy Landscape (Bubble = Weight)", fontsize=18)

# --- E. The Weights Table (Large Font) ---
ax5 = fig.add_subplot(gs[3, :])
ax5.axis('off')

table_data = portfolio[['symbol', 'weight', 'pred', 'avg_usd_vol']].copy()
table_data['weight'] = table_data['weight'].map('{:.2%}'.format)
table_data['pred'] = table_data['pred'].map('{:.4f}'.format)
table_data['avg_usd_vol'] = table_data['avg_usd_vol'].map('${:,.0f}'.format)
table_data.columns = ['Pair', 'Target Weight', 'Forecasted Return', '24h USD Volume']

tbl = ax5.table(cellText=table_data.values, colLabels=table_data.columns, 
                loc='center', cellLoc='center')

tbl.auto_set_font_size(False)
tbl.set_fontsize(16) # Main Table Font
tbl.scale(1, 3.2)    # Large vertical spacing

for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold', color='white', fontsize=18)
        cell.set_facecolor('#2c3e50')
    else:
        cell.set_facecolor('#ffffff')

# --- Header ---
myt_now = context_data['ts_local'].max()
current_time_myt = datetime.now(malaysia_tz)
plt.suptitle(f"Portfolio Deployment Snapshot | {myt_now.strftime('%A, %d %B %Y | %H:%M')} MYT\n\n(generated {current_time_myt.strftime('%A, %d %B %Y | %H:%M')} MYT)", 
             fontsize=24, fontweight='bold', y=0.98)

plt.tight_layout(rect=[0, 0.03, 1, 0.94])
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

res = pd.read_parquet(output_eval_file)
res.index = pd.to_datetime(res.ts)


fig, [ax1, ax2, ax3] = plt.subplots(3,1,figsize=(18,10), sharex=True)

# returns
((np.exp(res.strategy) - 1) * 100).rolling(30,min_periods=1).mean().plot(y='strategy',ax=ax1,color='gray',label='strategy')
((np.exp(res.perf) - 1) * 100).rolling(30,min_periods=1).mean().plot(y='perf',ax=ax1,color='black',label='alpha vs btc')
ax1.set_title('daily returns (30d sma)')
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(symbol='%'))
ax1.axhline(0, color='red', linestyle='--', alpha=0.5)
ax1.legend()

# equity multiplier
np.exp(res.equity).plot(y='equity',ax=ax2,color='blue',title='equity growth')
ax2.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:.2f}x'))
ax2.axhline(1, color='gray', linestyle='--', alpha=0.5)
ax2.set_yscale('log')

# fees
(res.fee * 10_000).rolling(10,min_periods=1).mean().plot(y='fee',ax=ax3,title='fees (10d sma)')
ax3.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:.0f}bps'))
plt.tight_layout()
plt.show()

In [ ]:
import scrapbook as sb
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

res = pl.read_parquet(output_eval_file)

# 1. Prepare Yearly Metrics
# We use the 'strategy' (log ret) and 'perf' (log excess ret) columns
yearly_stats = (
    res.with_columns(pl.col('ts').dt.year().alias('year'))
    .group_by('year')
    .agg([
        # Annualized IR: (Mean Excess / Std Excess) * sqrt(365)
        ((pl.col('perf').mean() / (pl.col('perf').std() + 1e-9)) * np.sqrt(365)).alias('ir'),
        
        # Max Drawdown: Min of (Equity / Running Max - 1)
        ((pl.col('strategy').cum_sum().exp() / 
          pl.col('strategy').cum_sum().exp().cum_max()) - 1).min().alias('mdd'),
          
        # Growth Multiplier: exp(sum of log returns)
        (pl.col('strategy').sum().exp()).alias('growth')
    ])
    .sort('year')
)

# 2. Calculate Total Period Metrics
total_ir = (res['perf'].mean() / (res['perf'].std() + 1e-9)) * np.sqrt(365)
total_equity = res['strategy'].cum_sum().exp()
total_mdd = ((total_equity / total_equity.cum_max()) - 1).min()
total_growth = total_equity.tail(1).item()

# 3. Record Metrics for Papermill (using Scrapbook)
sb.glue("ir_total", float(total_ir))
sb.glue("mdd_total", float(total_mdd))
sb.glue("growth_total", float(total_growth))
sb.glue("yearly_stats", yearly_stats.to_pandas().to_dict(orient='records'))

# 4. Yearly Bar Chart Visualization
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
y_pd = yearly_stats.to_pandas()
years = y_pd['year'].astype(str)

# IR Chart
axes[0].bar(years, y_pd['ir'], color='skyblue', edgecolor='black')
axes[0].set_title(f'Information Ratio (Total: {total_ir:.2f})')
axes[0].axhline(0, color='black', lw=1)

# MDD Chart
axes[1].bar(years, y_pd['mdd'] * 100, color='salmon', edgecolor='black')
axes[1].set_title(f'Max Drawdown (Total: {total_mdd*100:.1f}%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

# Growth Multiplier
axes[2].bar(years, y_pd['growth'], color='lightgreen', edgecolor='black')
axes[2].set_title(f'Yearly Growth (Total: {total_growth:.2f}x)')
axes[2].yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:.2f}x'))
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()